# Local Pipeline: Train → Generate → Evaluate

Run the full CFM pipeline locally at reduced scale. Uses the same `cfm.*` code
that runs on HPC — just with smaller configs (5 epochs, batch_size=64, 20 ODE steps).

For full-scale HPC runs, use `/submit` from claude-hpc.

In [ ]:
%matplotlib inline

import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Ensure cfm package is importable when running from notebooks/
sys.path.insert(0, os.path.abspath(".."))

from cfm.config import CFMConfig
from cfm.training.trainer import CFMTrainer
from cfm.model.sampler import CFMSampler
from cfm.model.vector_field import ConditionalVectorField
from cfm.evaluation.metrics import evaluate_all, acf_comparison
from cfm.evaluation.visualize import (
    plot_sample_paths,
    plot_diurnal_pattern,
    plot_marginal_distributions,
    plot_acf,
)

## Configuration

Reduced-scale config for local runs. Same architecture as HPC, just fewer
epochs/steps so it finishes in minutes on CPU.

In [ ]:
config = CFMConfig(
    harxhar_path="../data/all30min",
    # Reduced training scale
    num_epochs=5,
    batch_size=64,
    warmup_steps=50,
    checkpoint_every=2,
    # Faster sampling
    num_ode_steps=20,
    solver="euler",
    # Notebook-safe DataLoader settings
    num_workers=1,  # bypass auto-detect; 0 would trigger auto
    pin_memory=False,
    persistent_workers=False,
)

print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Epochs: {config.num_epochs}, Batch size: {config.batch_size}")
print(f"ODE steps: {config.num_ode_steps}, Solver: {config.solver}")

## Stage 1: Train

In [ ]:
trainer = CFMTrainer(config)

n_params = sum(p.numel() for p in trainer.model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Train batches: {len(trainer.train_loader)}, Val batches: {len(trainer.val_loader)}")
print(f"Test batches: {len(trainer.test_loader)}")

trainer.fit()

## Stage 2: Generate

Load the best checkpoint and generate synthetic proportions for the test set.

In [ ]:
# Load best checkpoint
ckpt_path = os.path.join(config.checkpoint_dir, "best.pt")
ckpt = torch.load(ckpt_path, map_location=trainer.device, weights_only=False)

model = ConditionalVectorField(
    output_dim=config.output_dim,
    cond_dim=config.cond_dim,
    hidden_dims=config.hidden_dims,
    time_embed_dim=config.time_embed_dim,
).to(trainer.device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Build sampler with diurnal prior
diurnal_mean = None
if ckpt.get("diurnal_mean") is not None:
    diurnal_mean = torch.tensor(ckpt["diurnal_mean"], dtype=torch.float32, device=trainer.device)

sampler = CFMSampler(
    vector_field=model,
    solver=config.solver,
    num_steps=config.num_ode_steps,
    diurnal_mean=diurnal_mean,
    diurnal_std=config.diurnal_prior_std,
)

print(f"Loaded checkpoint from epoch {ckpt['epoch']}, val_loss={ckpt['best_val_loss']:.6f}")

In [ ]:
# Generate proportions for the test set
all_gen_props = []
all_real_props = []
all_daily_rvs = []

for x_1, cond in trainer.test_loader:
    cond = cond.to(trainer.device)
    gen_props = sampler.sample_proportions(cond)
    all_gen_props.append(gen_props.cpu().numpy())
    all_real_props.append(x_1.numpy())
    # Extract daily_rv from conditioning (last element = sqrt(daily_rv))
    sqrt_rv = cond[:, -1].cpu().numpy()
    all_daily_rvs.append(sqrt_rv ** 2)

gen_props = np.concatenate(all_gen_props)
real_props = np.concatenate(all_real_props)
daily_rvs = np.concatenate(all_daily_rvs)

# Absolute RV = proportions * daily_rv
gen_abs = gen_props * daily_rvs[:, None]

print(f"Generated {len(gen_props)} days of synthetic proportions")
print(f"Real test set: {len(real_props)} days")

## Stage 3: Evaluate

In [ ]:
metrics = evaluate_all(real_props, gen_props, daily_rv=daily_rvs)

print("Evaluation Metrics")
print("=" * 40)
for k, v in metrics.items():
    print(f"  {k:30s} {v:.6f}")

## Visualizations

In [ ]:
plot_sample_paths(real_props, gen_props, n_samples=8)
plt.show()

In [ ]:
plot_diurnal_pattern(real_props, gen_props)
plt.show()

In [ ]:
plot_marginal_distributions(real_props, gen_props)
plt.show()

In [ ]:
acf = acf_comparison(real_props, gen_props)
plot_acf(acf["real_acf"], acf["gen_acf"])
plt.show()